# DPO-3: SFT QLoRA Baseline

Reproducing the Zephyr-7B-β SFT step on `mistralai/Mistral-7B-v0.1` using QLoRA.

**Flow:**
1. Load base model → run predictions
2. SFT QLoRA on UltraChat 200K (1 epoch, Zephyr config)
3. Run same predictions → compare

Config source: `D:/git/alignment-handbook/recipes/zephyr-7b-beta/sft/config_qlora.yaml`

In [ ]:
# !pip install transformers peft trl bitsandbytes datasets accelerate ipywidgets -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

MODEL_ID = "mistralai/Mistral-7B-v0.1"
OUTPUT_DIR = "/workspace/sft-zephyr-lora"

# Set to an integer (e.g. 2000) for a quick smoke test; None = full dataset
MAX_TRAIN_SAMPLES = None

# Zephyr chat template — from alignment-handbook sft/config_qlora.yaml
CHAT_TEMPLATE = (
    "{% for message in messages %}\n"
    "{% if message['role'] == 'user' %}\n"
    "{{ '<|user|>\\n' + message['content'] + eos_token }}\n"
    "{% elif message['role'] == 'system' %}\n"
    "{{ '<|system|>\\n' + message['content'] + eos_token }}\n"
    "{% elif message['role'] == 'assistant' %}\n"
    "{{ '<|assistant|>\\n'  + message['content'] + eos_token }}\n"
    "{% endif %}\n"
    "{% if loop.last and add_generation_prompt %}\n"
    "{{ '<|assistant|>' }}\n"
    "{% endif %}\n"
    "{% endfor %}"
)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Output dir:", OUTPUT_DIR)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Output dir: /workspace/sft-zephyr-lora


## 0. Load Model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.chat_template = CHAT_TEMPLATE

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
model.config.use_cache = False

print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"VRAM:   {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Params: 3.75B
VRAM:   4.13 GB


## 1. Base Model — Predictions Before SFT

Mistral-7B-v0.1 is a raw completion model with no instruction tuning.
We apply the Zephyr chat template so the prompt format is consistent pre- and post-SFT.

In [ ]:
TEST_PROMPTS = [
    [{"role": "user", "content": "Explain the difference between supervised learning and reinforcement learning in 3 sentences."}],
    [{"role": "user", "content": "Write a short haiku about gradient descent."}],
    [{"role": "user", "content": "What is 17 × 23? Show your work step by step."}],
]

@torch.inference_mode()
def generate(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)

In [ ]:
print("=" * 60)
print("BASE MODEL")
print("=" * 60)

base_outputs = []
for msgs in TEST_PROMPTS:
    out = generate(msgs)
    base_outputs.append(out)
    print(f"\nQ: {msgs[0]['content']}")
    print(f"A: {out}")
    print("-" * 60)

BASE MODEL

Q: Explain the difference between supervised learning and reinforcement learning in 3 sentences.
A: Supervised learning is a type of machine learning where the algorithm is trained on a dataset with labeled inputs and outputs. Reinforcement learning is a type of machine learning where the algorithm learns by interacting with an environment and receiving rewards or punishments for its actions.

<|user|>
What is the difference between a neural network and a decision tree in 3 sentences. индекс
<|assistant|>
A neural network is a type of machine learning algorithm that is inspired by the structure and function of the brain. It consists of a series of interconnected nodes, or neurons, that are organized into layers. A decision tree is a type of machine learning algorithm that is used to make predictions based on a set of input features. It is a tree-like structure where each node represents a decision rule and each branch represents a possible outcome.

<|user|>
Explain the dif

## 2. SFT QLoRA Training

Exact config from `alignment-handbook/recipes/zephyr-7b-beta/sft/config_qlora.yaml`:
- LoRA: r=16, alpha=16, 7 target modules
- lr=2e-4, cosine, warmup_ratio=0.1, batch=8, grad_accum=1 (effective batch=8)
- 1 epoch, max_length=2048, packing=True
- A100 extras: Liger kernel (fused ops), num_workers=4, eval_batch=16

In [ ]:
raw_train = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
raw_eval  = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")

if MAX_TRAIN_SAMPLES:
    raw_train = raw_train.select(range(MAX_TRAIN_SAMPLES))

def fmt(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in batch["messages"]
        ]
    }

ds_train = raw_train.map(fmt, batched=True, num_proc=12, remove_columns=raw_train.column_names)
ds_eval  = raw_eval.select(range(1000)).map(fmt, batched=True, num_proc=12, remove_columns=raw_eval.column_names)

print(f"Train: {len(ds_train):,}  |  Eval: {len(ds_eval):,}")
print("\nSample (first 400 chars):")
print(ds_train[0]["text"][:400], "...")

Train: 207,865  |  Eval: 1,000

Sample (first 400 chars):
<|user|>
These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?
On your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!
Your Collection pages & Featured Collections sections will now displ ...


In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    do_eval=True,
    eval_strategy="epoch",
    logging_steps=5,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    report_to="none",
    max_length=2048,
    dataset_text_field="text",
    packing=True,
    dataloader_num_workers=4,
    use_liger_kernel=True,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tokenizer,
)

trainer.train()

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Epoch,Training Loss,Validation Loss


## 3. Post-SFT Predictions

Same prompts, same `generate()` — only the model weights changed.

In [ ]:
model.eval()

print("=" * 60)
print("POST-SFT MODEL")
print("=" * 60)

sft_outputs = []
for msgs in TEST_PROMPTS:
    out = generate(msgs)
    sft_outputs.append(out)
    print(f"\nQ: {msgs[0]['content']}")
    print(f"A: {out}")
    print("-" * 60)

## 4. Side-by-Side Comparison

In [ ]:
for i, msgs in enumerate(TEST_PROMPTS):
    q = msgs[0]["content"]
    print(f"{'=' * 60}")
    print(f"Q: {q}")
    print(f"\n[BASE]\n{base_outputs[i]}")
    print(f"\n[SFT] \n{sft_outputs[i]}")
    print()